
# 1. Binär- und Hexadezimalzahlen – interaktives Notebook

**Lernziele**
- Du verstehst, was das **hexadezimale System (Basis 16)** ist und welche Zeichen dafür verwendet werden (0–9, A–F).
- Du kannst **Hex ↔ Binär** sicher umrechnen (jede Hex-Stelle entspricht **4 Bits**).
- Du kannst **Dezimal ↔ Binär** und **Dezimal ↔ Hex** umrechnen und den **Lösungsweg** nachvollziehen.
- Du trainierst mit einem **Übungstrainer** und überprüfst deine Lösungen automatisch.

**So arbeitest du**
1. Lies die kurzen Erklärungen.
2. Führe Code-Zellen nacheinander mit ▶️ aus.
3. Nutze die **interaktiven Widgets** (Eingabefelder, Buttons), um selbst zu experimentieren.

> **Hinweis:** In Google Colab zuerst **Laufzeit → Alles ausführen**. Die erste Zelle installiert/aktiviert bei Bedarf `ipywidgets` und den Widget‑Manager.

In [ ]:
#@title Setup starten { display-mode: "form" }
# Setup (funktioniert in Jupyter & Google Colab)
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

def _pip(package_spec: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_spec])

# ipywidgets sicherstellen
try:
    import ipywidgets as widgets
except Exception:
    _pip("ipywidgets>=8.1")
    import ipywidgets as widgets

# In Colab: Widget-Manager aktivieren (für stabile Darstellung)
if IN_COLAB:
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except Exception as e:
        print("Konnte Widget-Manager nicht aktivieren:", e)

from IPython.display import display
import math, random
print("Bereit. Colab:", IN_COLAB, "| ipywidgets:", widgets.__version__)


Bereit. Colab: True | ipywidgets: 7.7.1


## 2. Kurze Einführung

- **Hexadezimal (Basis 16):** Ziffern `0–9` und Buchstaben `A–F` (für 10–15).
- Jede Stelle steht für eine **Potenz von 16** (… 16², 16¹, 16⁰).
- Beispiel: `3F₁₆ = 3 × 16 + 15 = 48 + 15 = 63`.

**Hex ↔ Binär (wichtig!):**
- **1 Hex-Zeichen = 4 Bits (ein „Nibble“)**
- Beispiel 1: `2A₁₆` → `2 = 0010`, `A = 1010` → `2A₁₆ = 0010 1010₂`
- Beispiel 2: `1110 1101₂` → gruppiere `1110` und `1101` → `E` und `D` → `ED₁₆`

Unten findest du eine Tabelle `0–F` → Binär.


In [ ]:
#@title Tabelle Hex, Dec, Binär { display-mode: "form" }
def show_nibble_table():
    print("Hex  Dec  Binär")
    print("----------------")
    for i in range(16):
        h = format(i, "X")
        b = format(i, "04b")
        print(f"{h:>3}  {i:>3}  {b}")

show_nibble_table()


Hex  Dec  Binär
----------------
  0    0  0000
  1    1  0001
  2    2  0010
  3    3  0011
  4    4  0100
  5    5  0101
  6    6  0110
  7    7  0111
  8    8  1000
  9    9  1001
  A   10  1010
  B   11  1011
  C   12  1100
  D   13  1101
  E   14  1110
  F   15  1111


## Hilfsfunktionen: Umwandlungsfunktionen (Hex ↔ Binär ↔ Dezimal)

In [ ]:
#@title Hilfsfunktionen starten { display-mode: "form" }
from typing import List, Tuple

HEX_DIGITS = "0123456789ABCDEF"

def _clean_bin(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().replace(" ", "").replace("_", "")
    s = s[2:] if s.lower().startswith("0b") else s
    if s == "":
        return ""
    if not all(c in "01" for c in s):
        raise ValueError("Binärzahl darf nur 0 und 1 (plus optionale Leerzeichen/Unterstriche) enthalten.")
    return s

def _clean_hex(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().replace(" ", "").replace("_", "").upper()
    s = s[2:] if s.startswith("0X") else s
    if s == "":
        return ""
    if not all(c in HEX_DIGITS for c in s):
        raise ValueError("Hexadezimalzahl darf nur 0–9 und A–F enthalten.")
    return s

def binary_to_hex(bin_str: str) -> str:
    b = _clean_bin(bin_str)
    if b == "":
        return ""
    pad = (-len(b)) % 4
    b = "0"*pad + b
    hx_chars = []
    for i in range(0, len(b), 4):
        nibble = b[i:i+4]
        hx_chars.append(format(int(nibble, 2), "X"))
    # Entferne führende Nullen (aber nicht, wenn Ergebnis leer wäre)
    hx = "".join(hx_chars).lstrip("0")
    return hx if hx != "" else "0"

def hex_to_binary(hx: str, group_nibbles: bool = True) -> str:
    h = _clean_hex(hx)
    if h == "":
        return ""
    bits = "".join(format(int(c, 16), "04b") for c in h)
    if not group_nibbles:
        return bits.lstrip("0") or "0"
    # gruppiert in 4er-Gruppen
    grouped = " ".join(bits[i:i+4] for i in range(0, len(bits), 4))
    # führende 0000 Gruppen optional reduzieren
    groups = grouped.split()
    while len(groups) > 1 and groups[0] == "0000":
        groups.pop(0)
    return " ".join(groups)

def binary_to_decimal(bin_str: str) -> int:
    b = _clean_bin(bin_str)
    return int(b, 2) if b != "" else 0

def hex_to_decimal(hx: str) -> int:
    h = _clean_hex(hx)
    return int(h, 16) if h != "" else 0

def decimal_to_binary_steps(n: int) -> Tuple[str, List[Tuple[int, int, int]]]:
    if not isinstance(n, int) or n < 0:
        raise ValueError("Bitte eine nichtnegative ganze Zahl verwenden.")
    if n == 0:
        return "0", [(0, 0, 0)]
    steps = []
    digits = []
    x = n
    while x > 0:
        q, r = divmod(x, 2)
        steps.append((x, q, r))  # (aktuelles n, n//2, Rest)
        digits.append(str(r))
        x = q
    digits.reverse()
    return "".join(digits), steps

def decimal_to_hex_steps(n: int) -> Tuple[str, List[Tuple[int, int, int]]]:
    if not isinstance(n, int) or n < 0:
        raise ValueError("Bitte eine nichtnegative ganze Zahl verwenden.")
    if n == 0:
        return "0", [(0, 0, 0)]
    steps = []
    digits = []
    x = n
    while x > 0:
        q, r = divmod(x, 16)
        steps.append((x, q, r))  # (aktuelles n, n//16, Rest)
        digits.append(HEX_DIGITS[r])
        x = q
    digits.reverse()
    return "".join(digits), steps

def pretty_print_div2_steps(steps: List[Tuple[int, int, int]]) -> None:
    print("n   n//2  Rest")
    print("------------")
    for n, q, r in steps:
        print(f"{n:<3} {q:<5} {r}")
    print("(Reste von unten nach oben lesen)")

def pretty_print_div16_steps(steps: List[Tuple[int, int, int]]) -> None:
    print("n    n//16  Rest  Hex-Rest")
    print("---------------------------")
    for n, q, r in steps:
        print(f"{n:<4} {q:<6} {r:<4} {HEX_DIGITS[r]}")
    print("(Hex-Ziffern von unten nach oben lesen)")


## 3. Interaktiv: Hex ↔ Binär

- Tippe eine **Hex-Zahl** ein und sieh sofort die **Binärdarstellung**.
- Oder tippe **Binär** und sieh die **Hex-Darstellung**.

In [ ]:
#@title Interaktiv: Hex - Binär starten { display-mode: "form" }
from ipywidgets import interact

@interact(hex_string="A9E", group_nibbles=True)
def hex_to_binary_widget(hex_string, group_nibbles):
    try:
        b = hex_to_binary(hex_string, group_nibbles=group_nibbles)
        print("Binär:", b if b else "(leer)")
    except Exception as e:
        print("Fehler:", e)

@interact(binary_string="1010 1001 1110")
def binary_to_hex_widget(binary_string):
    try:
        h = binary_to_hex(binary_string)
        print("Hex:", h if h else "(leer)")
    except Exception as e:
        print("Fehler:", e)


interactive(children=(Text(value='A9E', description='hex_string'), Checkbox(value=True, description='group_nib…

interactive(children=(Text(value='1010 1001 1110', description='binary_string'), Output()), _dom_classes=('wid…

## 4. Interaktiv: Dezimal → Binär / Hex **mit Rechenschritten**

### Dezimal → Binär:
Das Standardverfahren ist **wiederholtes Teilen durch 2**. Reste werden **von unten nach oben** gelesen.

### Dezimal → Hex:
Auch hier: **wiederholtes Teilen**, diesmal durch **16**. Die Hex-Ziffern der Reste werden **von unten nach oben** gelesen.

In [ ]:
#@title Interaktiv Dezimal -> Binär/Hex starten { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, Markdown

int_box = widgets.BoundedIntText(
    value=174, min=0, max=10_000_000, step=1, description="Dezimal:"
)
out_bin = widgets.Output()
out_hex = widgets.Output()

def update_steps(change=None):
    out_bin.clear_output()
    out_hex.clear_output()
    n = int_box.value
    bstr, steps2 = decimal_to_binary_steps(n)
    hstr, steps16 = decimal_to_hex_steps(n)
    with out_bin:
        print("Binär:", bstr)
        pretty_print_div2_steps(steps2)
    with out_hex:
        print("Hex:", hstr)
        pretty_print_div16_steps(steps16)

int_box.observe(update_steps, names="value")
update_steps(None)

display(widgets.VBox([int_box, widgets.HBox([out_bin, out_hex])]))


## 5. Interaktiv: **Drei‑Wege‑Umrechner** (Dezimal ↔ Hex ↔ Binär)

In [ ]:
#@title Dreo-Wege-Umrechner starten { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display

dec_in = widgets.Text(value="174", description="Dezimal:")
hex_in = widgets.Text(value="AE", description="Hex:")
bin_in = widgets.Text(value="1010 1110", description="Binär:")
msg = widgets.HTML(value="")

_lock = {"busy": False}

def _set_silent(widget, value):
    # set without triggering endless loops
    widget.unobserve_all()
    widget.value = value
    widget.observe(on_change, names="value")

def on_change(change):
    if _lock["busy"]:
        return
    _lock["busy"] = True
    try:
        src = change["owner"]
        if src is dec_in:
            # update hex/bin
            try:
                n = int(dec_in.value.strip() or "0")
                hx, _ = decimal_to_hex_steps(n)
                b, _ = decimal_to_binary_steps(n)
                _set_silent(hex_in, hx)
                # gruppieren in Nibbles
                bits = hex_to_binary(hx, group_nibbles=True)
                _set_silent(bin_in, bits)
                msg.value = ""
            except Exception as e:
                msg.value = f"<span style='color:#b00'>Fehler: {e}</span>"
        elif src is hex_in:
            try:
                hx = hex_in.value
                n = hex_to_decimal(hx)
                b = hex_to_binary(hx, group_nibbles=True)
                _set_silent(dec_in, str(n))
                _set_silent(bin_in, b)
                msg.value = ""
            except Exception as e:
                msg.value = f"<span style='color:#b00'>Fehler: {e}</span>"
        elif src is bin_in:
            try:
                b = bin_in.value
                n = binary_to_decimal(b)
                h = binary_to_hex(b)
                _set_silent(dec_in, str(n))
                _set_silent(hex_in, h)
                # ggf. Nibbles neu gruppieren
                _set_silent(bin_in, hex_to_binary(h, group_nibbles=True))
                msg.value = ""
            except Exception as e:
                msg.value = f"<span style='color:#b00'>Fehler: {e}</span>"
    finally:
        _lock["busy"] = False

for w in (dec_in, hex_in, bin_in):
    w.observe(on_change, names="value")

# Initial auslösen
on_change({"owner": dec_in})

display(widgets.VBox([dec_in, hex_in, bin_in, msg]))


## 6. Übungstrainer (mit sofortigem Feedback)

Wähle einen Aufgabentyp, klicke **Neue Aufgabe**, trage deine Antwort ein und klicke **Prüfen**.  
Der Trainer zählt richtige/falsche Antworten mit.


In [ ]:
#@title Übungstrainer starten { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import random  # sicherstellen, dass random importiert ist

KIND_OPTIONS = [
    "Dezimal → Binär",
    "Dezimal → Hex",
    "Binär → Dezimal",
    "Hex → Dezimal",
    "Binär → Hex",
    "Hex → Binär",
]

state = {
    "kind": KIND_OPTIONS[0],
    "question": "",
    "expected": "",
    "right": 0,
    "wrong": 0,
}

# Helfer, der Binär-Strings in Nibbles erzeugt
# und garantiert, dass das erste Nibble ≠ 0000 ist.
def _rand_bits_nibbles(n_nibbles: int):
    first = format(random.randint(1, 15), "04b")  # 0001..1111
    if n_nibbles > 1:
        rest = "".join(format(random.randint(0, 15), "04b") for _ in range(n_nibbles - 1))
    else:
        rest = ""
    bits = first + rest
    bits_g = " ".join(bits[i:i+4] for i in range(0, len(bits), 4))
    return bits, bits_g

def _new_task(kind: str):
    if kind == "Dezimal → Binär":
        n = random.randint(0, 255)
        return f"Wandle {n}₁₀ in Binär um:", format(n, "b")

    if kind == "Dezimal → Hex":
        n = random.randint(0, 4095)
        return f"Wandle {n}₁₀ in Hex um:", format(n, "X")

    if kind == "Binär → Dezimal":
        # Nur volle Nibbles, erstes Nibble garantiert ≠ 0000
        nibbles = random.randint(1, 3)  # 4/8/12 Bit
        bits, bits_g = _rand_bits_nibbles(nibbles)
        return f"Wandle {bits_g}₂ in Dezimal um:", str(int(bits, 2))

    if kind == "Hex → Dezimal":
        # 1–3 Hex-Ziffern
        n = random.randint(0, 0xFFF)
        h = format(n, "X")
        return f"Wandle {h}₁₆ in Dezimal um:", str(int(h, 16))

    if kind == "Binär → Hex":
        # Nur volle Nibbles, erstes Nibble garantiert ≠ 0000
        nibbles = random.randint(1, 3)  # 4/8/12 Bit
        bits, bits_g = _rand_bits_nibbles(nibbles)
        hx = format(int(bits, 2), "X")
        return f"Wandle {bits_g}₂ in Hex um:", hx

    if kind == "Hex → Binär":
        # 1–3 Hex-Ziffern → 4/8/12 Bits (hier bleibt Verhalten wie zuvor)
        n = random.randint(0, 0xFFF)
        h = format(n, "X")
        bits = "".join(format(int(c, 16), "04b") for c in h)
        bits_g = " ".join(bits[i:i+4] for i in range(0, len(bits), 4))
        return f"Wandle {h}₁₆ in Binär um:", bits_g

    raise ValueError("Unbekannter Aufgabentyp")

# Widgets
kind_dd = widgets.Dropdown(options=KIND_OPTIONS, value=KIND_OPTIONS[0], description="Aufgabe:")
new_btn = widgets.Button(description="Neue Aufgabe")
check_btn = widgets.Button(description="Prüfen")
answer = widgets.Text(description="Antwort:")
feedback = widgets.Output()
score = widgets.HTML()

def _update_score():
    score.value = f"<b>Punkte:</b> ✔ {state['right']} &nbsp;&nbsp; ✖ {state['wrong']}"

def on_new_clicked(_=None):
    feedback.clear_output()
    state["kind"] = kind_dd.value
    q, exp = _new_task(state["kind"])
    state["question"] = q
    state["expected"] = exp.upper().strip()
    answer.value = ""
    with feedback:
        print(q)
    _update_score()

def on_check_clicked(_=None):
    with feedback:
        print("—"*30)
    user = answer.value.strip().replace(" ", "").replace("_", "").upper()
    exp = state["expected"]
    kind = state["kind"]

    # Hex-Antworten: '0x' zulassen + führende Nullen ignorieren
    if kind in ("Dezimal → Hex", "Binär → Hex"):
        if user == "":
            ok = False
        else:
            user_cmp = user[2:] if user.startswith("0X") else user
            user_cmp = user_cmp.lstrip("0") or "0"
            exp_cmp  = exp.replace(" ", "").replace("_", "")
            exp_cmp  = exp_cmp.lstrip("0") or "0"
            ok = (user_cmp == exp_cmp)

    # Binär-Antworten: '0b' zulassen + Leerzeichen/Unterstriche/führende Nullen ignorieren
    elif kind in ("Dezimal → Binär", "Hex → Binär"):
        if user == "":
            ok = False
        else:
            user_cmp = user[2:] if user.startswith("0B") else user
            user_cmp = user_cmp.lstrip("0") or "0"
            exp_cmp  = exp.replace(" ", "").replace("_", "")
            exp_cmp  = exp_cmp.lstrip("0") or "0"
            ok = (user_cmp == exp_cmp)

    # Dezimal-Antworten: führende Nullen erlauben
    elif kind in ("Binär → Dezimal", "Hex → Dezimal"):
        if user == "":
            ok = False
        else:
            try:
                ok = (int(user) == int(exp))
            except ValueError:
                ok = False

    else:
        ok = (user == exp)

    with feedback:
        if ok:
            print("✅ Richtig! Erwartet:", exp)
            state["right"] += 1
        else:
            print(f"❌ Leider falsch. Erwartet: {exp}, erhalten: {answer.value or '(leer)'}")
            state["wrong"] += 1
    _update_score()

new_btn.on_click(on_new_clicked)
check_btn.on_click(on_check_clicked)

_update_score()
on_new_clicked()

ui = widgets.VBox([
    kind_dd,
    widgets.HBox([new_btn, check_btn, score]),
    answer,
    feedback
])
display(ui)


Copyright 2025 [Markus Ineichen](mailto:markus.ineichen1@sluz.ch)

Code licence: [MIT License](https://mit-license.org/)

Text license: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)